# 02 — Clean & Build Analysis Views

Takes the raw DuckDB tables from 01-ingest and:
1. Filters to leading causes only (# prefix = rankable, non-overlapping)
2. Cleans cause names for display
3. Standardizes age group labels
4. Builds aggregated views:
   - `mortality_national` — all causes summed across race and sex
   - `mortality_female` — female-only totals by cause
   - `mortality_female_repro` — female 15–44 only by cause
   - `mortality_by_sex_age` — both sexes by age band and cause
5. Runs quality checks

In [ ]:
import sys, os
from pathlib import Path

PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

import pandas as pd
import duckdb

from src.ingest import load_config
from src.clean_quality import (
    get_connection, load_to_duckdb, run_sql,
    quality_report, save_interim, register_source
)

cfg = load_config('config.yaml')
con = get_connection(cfg)
print(f'Project: {cfg["project_name"]}')
print(f'DuckDB: {cfg["settings"]["duckdb_file"]}')

# Verify tables exist
tables = con.execute('SHOW TABLES').df()['name'].tolist()
print(f'Tables: {tables}')

## Cause name cleaning

The ICD-10 113 Cause List names include ICD codes in parentheses.
We create a clean display name for charts and exports.

In [ ]:
import re

def clean_cause_name(raw_name: str) -> str:
    """Remove # prefix and ICD codes from cause name for display.
    
    '#Diseases of heart (I00-I09,I11,I13,I20-I51)' -> 'Diseases of heart'
    """
    name = raw_name.strip()
    # Remove leading #
    if name.startswith('#'):
        name = name[1:]
    # Remove ICD codes in parentheses at end
    name = re.sub(r'\s*\([A-Z*][^)]+\)\s*$', '', name)
    return name.strip()

# Test
test_cases = [
    '#Diseases of heart (I00-I09,I11,I13,I20-I51)',
    '#Malignant neoplasms (C00-C97)',
    '#Accidents (unintentional injuries) (V01-X59,Y85-Y86)',
    '#Intentional self-harm (suicide) (*U03,X60-X84,Y87.0)',
    '#Assault (homicide) (*U01-*U02,X85-Y09,Y87.1)',
    '#Pregnancy, childbirth and the puerperium (O00-O99)',
]
for tc in test_cases:
    print(f'  {tc[:55]:<55} -> {clean_cause_name(tc)}')

## Build: `mortality_national`

Aggregate the race×sex table across race and sex to get national totals.
Keep only # causes (leading/rankable causes, mutually exclusive).

In [ ]:
df_national = run_sql("""
    SELECT
        icd_10_113_cause_list_code AS cause_code,
        icd_10_113_cause_list AS cause_raw,
        SUM(deaths) AS deaths,
        SUM(population) AS population
    FROM mortality_race_sex
    WHERE icd_10_113_cause_list LIKE '#%'
      AND single_race_6 != 'Not Available'
    GROUP BY cause_code, cause_raw
    ORDER BY deaths DESC
""", con)

# Note: population is summed across all race×sex combos for each cause,
# but since each cause row has the same population for a given race×sex,
# summing gives us total US population repeated per cause.
# We'll fix this: population should be a single value (total US pop).
# Actually for rate calculation we need the total population once.

# Get total US population (sum of distinct race×sex population values)
total_pop = run_sql("""
    SELECT SUM(population) as total_pop FROM (
        SELECT DISTINCT single_race_6, sex, population
        FROM mortality_race_sex
        WHERE icd_10_113_cause_list_code = 'GR113-054'
          AND population IS NOT NULL
          AND single_race_6 != 'Not Available'
    )
""", con).iloc[0, 0]

df_national['population'] = total_pop
df_national['cause'] = df_national['cause_raw'].apply(clean_cause_name)
df_national['crude_rate'] = df_national['deaths'] / df_national['population'] * 100_000
df_national['rank'] = range(1, len(df_national) + 1)

print(f'National leading causes: {len(df_national)} causes')
print(f'Total US population: {total_pop:,.0f}')
print()
df_national[['rank', 'cause', 'deaths', 'crude_rate']].head(10)

In [ ]:
# Save to DuckDB
load_to_duckdb(df_national[['cause_code', 'cause_raw', 'cause', 'deaths', 'population', 'crude_rate', 'rank']], 
               'mortality_national', con)
register_source(con, 'mortality_national', 'Derived: National totals',
    notes='Aggregated from mortality_race_sex. # causes only (leading, non-overlapping). Population = total US.')
print('✓ mortality_national saved')

## Build: `mortality_female`

Female-only totals by cause (all ages). From the race×sex table, filter to Female.

In [ ]:
# Female population (sum across races)
female_pop = run_sql("""
    SELECT SUM(population) as pop FROM (
        SELECT DISTINCT single_race_6, population
        FROM mortality_race_sex
        WHERE sex = 'Female'
          AND icd_10_113_cause_list_code = 'GR113-054'
          AND population IS NOT NULL
          AND single_race_6 != 'Not Available'
    )
""", con).iloc[0, 0]

df_female = run_sql("""
    SELECT
        icd_10_113_cause_list_code AS cause_code,
        icd_10_113_cause_list AS cause_raw,
        SUM(deaths) AS deaths
    FROM mortality_race_sex
    WHERE sex = 'Female'
      AND icd_10_113_cause_list LIKE '#%'
      AND single_race_6 != 'Not Available'
    GROUP BY cause_code, cause_raw
    ORDER BY deaths DESC
""", con)

df_female['population'] = female_pop
df_female['cause'] = df_female['cause_raw'].apply(clean_cause_name)
df_female['crude_rate'] = df_female['deaths'] / df_female['population'] * 100_000
df_female['rank'] = range(1, len(df_female) + 1)

print(f'Female leading causes: {len(df_female)} causes')
print(f'Female population (all ages): {female_pop:,.0f}')
print()
df_female[['rank', 'cause', 'deaths', 'crude_rate']].head(10)

In [ ]:
load_to_duckdb(df_female[['cause_code', 'cause_raw', 'cause', 'deaths', 'population', 'crude_rate', 'rank']],
               'mortality_female', con)
register_source(con, 'mortality_female', 'Derived: Female totals (all ages)',
    notes='Aggregated from mortality_race_sex, Female only. # causes only.')
print('✓ mortality_female saved')

## Build: `mortality_female_repro`

**The key table.** Female ages 15–44 only — reproductive age.
Uses the sex×age file (no race dimension needed).

In [ ]:
REPRO_AGES = [
    '15-19 years', '20-24 years', '25-29 years',
    '30-34 years', '35-39 years', '40-44 years'
]
repro_filter = ','.join(f"'{a}'" for a in REPRO_AGES)

# Female 15-44 population
female_repro_pop = run_sql(f"""
    SELECT SUM(population) as pop
    FROM mortality_sex_age
    WHERE sex = 'Female'
      AND icd_10_113_cause_list_code = 'GR113-054'
      AND five_year_age_groups IN ({repro_filter})
      AND population IS NOT NULL
""", con).iloc[0, 0]

df_female_repro = run_sql(f"""
    SELECT
        icd_10_113_cause_list_code AS cause_code,
        icd_10_113_cause_list AS cause_raw,
        SUM(deaths) AS deaths
    FROM mortality_sex_age
    WHERE sex = 'Female'
      AND five_year_age_groups IN ({repro_filter})
      AND icd_10_113_cause_list LIKE '#%'
    GROUP BY cause_code, cause_raw
    ORDER BY deaths DESC
""", con)

df_female_repro['population'] = female_repro_pop
df_female_repro['cause'] = df_female_repro['cause_raw'].apply(clean_cause_name)
df_female_repro['crude_rate'] = df_female_repro['deaths'] / df_female_repro['population'] * 100_000
df_female_repro['rank'] = range(1, len(df_female_repro) + 1)

print(f'Female 15-44 leading causes: {len(df_female_repro)} causes')
print(f'Female 15-44 population: {female_repro_pop:,.0f}')
print()
df_female_repro[['rank', 'cause', 'deaths', 'crude_rate']].head(10)

In [ ]:
load_to_duckdb(df_female_repro[['cause_code', 'cause_raw', 'cause', 'deaths', 'population', 'crude_rate', 'rank']],
               'mortality_female_repro', con)
register_source(con, 'mortality_female_repro', 'Derived: Female 15-44 (reproductive age)',
    notes='From mortality_sex_age, Female, ages 15-44. # causes only. Key analysis table.')
print('✓ mortality_female_repro saved')

## Build: `mortality_by_sex_age`

Both sexes by age band and cause — useful for the age distribution chart
showing where deaths concentrate by age vs where abortions concentrate.

In [ ]:
df_sex_age = run_sql("""
    SELECT
        sex,
        five_year_age_groups AS age_group,
        icd_10_113_cause_list_code AS cause_code,
        icd_10_113_cause_list AS cause_raw,
        deaths,
        population,
        crude_rate
    FROM mortality_sex_age
    WHERE icd_10_113_cause_list LIKE '#%'
      AND five_year_age_groups != 'Not Stated'
      AND deaths IS NOT NULL
    ORDER BY sex, five_year_age_groups, deaths DESC
""", con)

df_sex_age['cause'] = df_sex_age['cause_raw'].apply(clean_cause_name)

print(f'Sex × Age × Cause rows: {len(df_sex_age):,}')
print(f'Sex values: {df_sex_age["sex"].unique().tolist()}')
print(f'Age groups: {df_sex_age["age_group"].nunique()}')
print(f'Causes: {df_sex_age["cause"].nunique()}')
df_sex_age.head(5)

In [ ]:
load_to_duckdb(df_sex_age, 'mortality_by_sex_age', con)
register_source(con, 'mortality_by_sex_age', 'Derived: Sex × Age × Cause (leading only)',
    notes='From mortality_sex_age. # causes only. Excludes Not Stated age.')
print('✓ mortality_by_sex_age saved')

## Build: Race/Ethnicity Tables (NH White, NH Black, Hispanic)

Uses the `mortality_race_ethnicity` table (Sex × Hispanic Origin × Single Race 6 × Cause)
to build proper Non-Hispanic White, Non-Hispanic Black, and Hispanic aggregations
aligned with Guttmacher's race categories.

Display names come from `cause_display_names` table — no hardcoded mappings.


In [ ]:
# Load display name mapping from DuckDB
display_map = run_sql("SELECT cause_cdc, cause_display FROM cause_display_names", con)
display_dict = dict(zip(display_map['cause_cdc'], display_map['cause_display']))
print(f'Loaded {len(display_dict)} display name mappings from DuckDB')

def apply_display_names(df, col='cause'):
    """Apply display names from the mapping table. Falls back to raw name if no mapping."""
    df[col] = df[col].map(lambda x: display_dict.get(x, x))
    return df


In [ ]:
# --- NH White ---
df_nh_white = run_sql("""
    SELECT
        icd_10_113_cause_list_code AS cause_code,
        icd_10_113_cause_list AS cause_raw,
        SUM(deaths) AS deaths,
        SUM(CASE WHEN sex = 'Male' THEN deaths ELSE 0 END) AS male_deaths,
        SUM(CASE WHEN sex = 'Female' THEN deaths ELSE 0 END) AS female_deaths
    FROM mortality_race_ethnicity
    WHERE hispanic_origin = 'Not Hispanic or Latino'
      AND single_race_6 = 'White'
      AND icd_10_113_cause_list LIKE '#%'
    GROUP BY icd_10_113_cause_list_code, icd_10_113_cause_list
    ORDER BY deaths DESC
""", con)

# Get NH White population
nh_white_pop = run_sql("""
    SELECT SUM(population) as pop FROM (
        SELECT DISTINCT sex, population
        FROM mortality_race_ethnicity
        WHERE hispanic_origin = 'Not Hispanic or Latino'
          AND single_race_6 = 'White'
          AND icd_10_113_cause_list_code = 'GR113-054'
    )
""", con).iloc[0, 0]

df_nh_white['population'] = nh_white_pop
df_nh_white['crude_rate'] = (df_nh_white['deaths'] / nh_white_pop * 100_000).round(1)
df_nh_white['rank'] = range(1, len(df_nh_white) + 1)
df_nh_white['cause'] = df_nh_white['cause_raw'].apply(clean_cause_name)
df_nh_white = apply_display_names(df_nh_white, 'cause')

load_to_duckdb(df_nh_white, 'mortality_nh_white', con)
register_source(con, 'mortality_nh_white', 'Derived: NH White mortality (all causes)',
    notes='From mortality_race_ethnicity. NH White only. # causes only. Pop=195.4M.')
print(f'✓ mortality_nh_white: {len(df_nh_white)} causes, pop={nh_white_pop:,.0f}')
print(f'  Top 3: {df_nh_white.head(3)[["cause", "deaths", "crude_rate"]].to_string(index=False)}')


In [ ]:
# --- NH Black ---
df_nh_black = run_sql("""
    SELECT
        icd_10_113_cause_list_code AS cause_code,
        icd_10_113_cause_list AS cause_raw,
        SUM(deaths) AS deaths,
        SUM(CASE WHEN sex = 'Male' THEN deaths ELSE 0 END) AS male_deaths,
        SUM(CASE WHEN sex = 'Female' THEN deaths ELSE 0 END) AS female_deaths
    FROM mortality_race_ethnicity
    WHERE hispanic_origin = 'Not Hispanic or Latino'
      AND single_race_6 = 'Black or African American'
      AND icd_10_113_cause_list LIKE '#%'
    GROUP BY icd_10_113_cause_list_code, icd_10_113_cause_list
    ORDER BY deaths DESC
""", con)

nh_black_pop = run_sql("""
    SELECT SUM(population) as pop FROM (
        SELECT DISTINCT sex, population
        FROM mortality_race_ethnicity
        WHERE hispanic_origin = 'Not Hispanic or Latino'
          AND single_race_6 = 'Black or African American'
          AND icd_10_113_cause_list_code = 'GR113-054'
    )
""", con).iloc[0, 0]

df_nh_black['population'] = nh_black_pop
df_nh_black['crude_rate'] = (df_nh_black['deaths'] / nh_black_pop * 100_000).round(1)
df_nh_black['rank'] = range(1, len(df_nh_black) + 1)
df_nh_black['cause'] = df_nh_black['cause_raw'].apply(clean_cause_name)
df_nh_black = apply_display_names(df_nh_black, 'cause')

load_to_duckdb(df_nh_black, 'mortality_nh_black', con)
register_source(con, 'mortality_nh_black', 'Derived: NH Black mortality (all causes)',
    notes='From mortality_race_ethnicity. NH Black only. # causes only. Pop=43.0M.')
print(f'✓ mortality_nh_black: {len(df_nh_black)} causes, pop={nh_black_pop:,.0f}')
print(f'  Top 3: {df_nh_black.head(3)[["cause", "deaths", "crude_rate"]].to_string(index=False)}')


In [ ]:
# --- Hispanic (any race) ---
df_hispanic = run_sql("""
    SELECT
        icd_10_113_cause_list_code AS cause_code,
        icd_10_113_cause_list AS cause_raw,
        SUM(deaths) AS deaths,
        SUM(CASE WHEN sex = 'Male' THEN deaths ELSE 0 END) AS male_deaths,
        SUM(CASE WHEN sex = 'Female' THEN deaths ELSE 0 END) AS female_deaths
    FROM mortality_race_ethnicity
    WHERE hispanic_origin = 'Hispanic or Latino'
      AND single_race_6 != 'Not Available'
      AND icd_10_113_cause_list LIKE '#%'
    GROUP BY icd_10_113_cause_list_code, icd_10_113_cause_list
    ORDER BY deaths DESC
""", con)

# Hispanic population: sum across all race sub-groups
hisp_pop = run_sql("""
    SELECT SUM(population) as pop FROM (
        SELECT DISTINCT sex, single_race_6, population
        FROM mortality_race_ethnicity
        WHERE hispanic_origin = 'Hispanic or Latino'
          AND single_race_6 != 'Not Available'
          AND icd_10_113_cause_list_code = 'GR113-054'
    )
""", con).iloc[0, 0]

df_hispanic['population'] = hisp_pop
df_hispanic['crude_rate'] = (df_hispanic['deaths'] / hisp_pop * 100_000).round(1)
df_hispanic['rank'] = range(1, len(df_hispanic) + 1)
df_hispanic['cause'] = df_hispanic['cause_raw'].apply(clean_cause_name)
df_hispanic = apply_display_names(df_hispanic, 'cause')

load_to_duckdb(df_hispanic, 'mortality_hispanic', con)
register_source(con, 'mortality_hispanic', 'Derived: Hispanic mortality (all causes)',
    notes='From mortality_race_ethnicity. Hispanic (any race). # causes only. Pop=68.1M.')
print(f'✓ mortality_hispanic: {len(df_hispanic)} causes, pop={hisp_pop:,.0f}')
print(f'  Top 3: {df_hispanic.head(3)[["cause", "deaths", "crude_rate"]].to_string(index=False)}')


In [ ]:
# --- cause_detail_by_sex_race: sub-cause breakdowns for exploratory charts ---
# Pulls accident subtypes, suicide methods, and homicide methods from the
# original race×sex table for use in 04-viz exploratory analysis.

DETAIL_CAUSES = {
    'accidents_subtype': [
        'GR113-114',  # Motor vehicle accidents
        'GR113-118',  # Falls
        'GR113-119',  # Accidental discharge of firearms
        'GR113-120',  # Accidental drowning
        'GR113-122',  # Accidental poisoning (overdoses)
    ],
    'accidents_total': ['GR113-112'],  # All accidents
    'suicide_subtype': [
        'GR113-125',  # Suicide by firearms
    ],
    'suicide_total': ['GR113-124'],  # All suicide
    'homicide_subtype': [
        'GR113-128',  # Homicide by firearms
    ],
    'homicide_total': ['GR113-127'],  # All homicide
}

# Build a flat list of (category, code) pairs
code_to_cat = {}
for cat, codes in DETAIL_CAUSES.items():
    for code in codes:
        code_to_cat[code] = cat

all_codes = list(code_to_cat.keys())
codes_str = ','.join(f"'{c}'" for c in all_codes)

df_detail = run_sql(f"""
    SELECT
        icd_10_113_cause_list_code,
        icd_10_113_cause_list,
        sex,
        single_race_6,
        deaths,
        population
    FROM mortality_race_sex
    WHERE icd_10_113_cause_list_code IN ({codes_str})
""", con)

df_detail['category'] = df_detail['icd_10_113_cause_list_code'].map(code_to_cat)

load_to_duckdb(df_detail, 'cause_detail_by_sex_race', con)
register_source(con, 'cause_detail_by_sex_race',
    'Derived: Sub-cause detail for accidents/suicide/homicide',
    notes='From mortality_race_sex. Sub-causes by sex and race for exploratory charts (04-viz).')
print(f'✓ cause_detail_by_sex_race: {len(df_detail)} rows, categories: {sorted(df_detail["category"].unique())}')


## Quality Report

In [ ]:
print('── mortality_national ──')
qr_nat = quality_report(df_national, 'mortality_national', con,
    required_columns=['cause', 'deaths', 'population', 'crude_rate', 'rank'])

print('\n── mortality_female ──')
qr_fem = quality_report(df_female, 'mortality_female', con,
    required_columns=['cause', 'deaths', 'population', 'crude_rate', 'rank'])

print('\n── mortality_female_repro ──')
qr_repro = quality_report(df_female_repro, 'mortality_female_repro', con,
    required_columns=['cause', 'deaths', 'population', 'crude_rate', 'rank'])

## Key Validation: The Headline Numbers

In [ ]:
# Abortion count
abortion_total = 1_124_000  # Guttmacher 2024
abortion_15_44 = 1_121_752  # Excluding <15 age group

print('══════════════════════════════════════════════════════════')
print('  KEY FINDINGS: "What if abortion were a cause of death?"')
print('══════════════════════════════════════════════════════════')
print()

# National
nat_1 = df_national.iloc[0]
print(f'NATIONAL (all persons):')
print(f'  #1 cause: {nat_1["cause"]} ({nat_1["deaths"]:,.0f})')
print(f'  Abortion: {abortion_total:,.0f}')
print(f'  → Abortion would be #{1} — {abortion_total/nat_1["deaths"]:.2f}× the current #1')
print()

# Female all ages
fem_1 = df_female.iloc[0]
print(f'FEMALE (all ages):')
print(f'  #1 cause: {fem_1["cause"]} ({fem_1["deaths"]:,.0f})')
print(f'  Abortion: {abortion_total:,.0f}')
print(f'  → Abortion would be #{1} — {abortion_total/fem_1["deaths"]:.2f}× the current #1')
print()

# Female 15-44 — THE BIG ONE
repro_1 = df_female_repro.iloc[0]
print(f'FEMALE 15-44 (reproductive age):')
print(f'  #1 cause: {repro_1["cause"]} ({repro_1["deaths"]:,.0f})')
print(f'  Abortion: {abortion_15_44:,.0f}')
print(f'  → Abortion would be #{1} — {abortion_15_44/repro_1["deaths"]:.1f}× the current #1')
print()
print('══════════════════════════════════════════════════════════')

## Save interim Parquet files

In [ ]:
save_interim(df_national[['cause_code', 'cause', 'deaths', 'population', 'crude_rate', 'rank']],
             cfg, 'mortality_national.parquet')
save_interim(df_female[['cause_code', 'cause', 'deaths', 'population', 'crude_rate', 'rank']],
             cfg, 'mortality_female.parquet')
save_interim(df_female_repro[['cause_code', 'cause', 'deaths', 'population', 'crude_rate', 'rank']],
             cfg, 'mortality_female_repro.parquet')
save_interim(df_sex_age[['sex', 'age_group', 'cause_code', 'cause', 'deaths', 'population', 'crude_rate']],
             cfg, 'mortality_by_sex_age.parquet')

## Final table inventory

In [ ]:
print('\n── DuckDB Tables ──')
tables = con.execute('SHOW TABLES').df()
for t in sorted(tables['name'].tolist()):
    cnt = con.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]
    print(f'  {t:<30} {cnt:>8,} rows')

con.close()
print('\n✓ Cleaning complete.')

---
**Next:** open `03-prepare.ipynb` to build the "Without" / "With" comparison tables and export.